# Amazon ML Challenge 2026




In [18]:
# 1 — Install packages and inspect Colab resources

!pip -q install polars pyarrow rapidfuzz

import os
import re
import math
import time
import random
import shutil
import unicodedata
import subprocess

from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import polars as pl

print("CPU cores :", os.cpu_count())


disk = shutil.disk_usage("/content")
print("Disk total:", round(disk.total / 1024**3, 2), "GB")
print("Disk free :", round(disk.free / 1024**3, 2), "GB")


CPU cores : 2
Disk total: 107.72 GB
Disk free : 80.49 GB


In [2]:
# 2 — Mount Google Drive

from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [20]:
# 3 — Fixed dataset location and separate persistent output folder
MY_DRIVE = Path('/content/drive/MyDrive')
DATASET_ZIP = MY_DRIVE / 'Amazon_Ml_Challange_2026' / 'Data' / 'Raw' / 'dataset.zip'
DRIVE_EXTRACT_DIR = DATASET_ZIP.parent.parent / 'Processed' / 'dataset'
DRIVE_PROJECT = MY_DRIVE / 'Amazon_ML_Challenge'
LOCAL_ROOT = Path('/content/amazon_ml_challenge')
EXTRACT_DIR = LOCAL_ROOT / 'extracted'

if not DATASET_ZIP.is_file():
    raise FileNotFoundError(
        f'Archive not visible at {DATASET_ZIP}. If this is a shared folder, '
        'add a shortcut to Amazon_Ml_Challange_2026 in My Drive, or edit DATASET_ZIP above.'
    )
for folder in ('reports', 'processed', 'candidates', 'models', 'submissions'):
    (DRIVE_PROJECT / folder).mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print('Input archive:', DATASET_ZIP)
print('Archive size (GiB):', round(DATASET_ZIP.stat().st_size / 1024**3, 2))
print('Persistent TSV copies:', DRIVE_EXTRACT_DIR)
print('Persistent reports:', DRIVE_PROJECT)
print('Local working directory:', LOCAL_ROOT)


Input archive: /content/drive/MyDrive/Amazon_Ml_Challange_2026/Data/Raw/dataset.zip
Archive size (GiB): 1.02
Persistent TSV copies: /content/drive/MyDrive/Amazon_Ml_Challange_2026/Data/Processed/dataset
Persistent reports: /content/drive/MyDrive/Amazon_ML_Challenge
Local working directory: /content/amazon_ml_challenge


## Extract the `train/` and `test/` folders

Copy the ZIP once to Colab's local disk and extract only the seven required TSVs. The extraction cell prints the ZIP member paths before reading the data, so unexpected names or folders fail clearly. The extracted files are also copied to `Data/Processed/dataset/train` and `Data/Processed/dataset/test` in Drive. Reruns skip copies with matching sizes. The local copies support faster EDA.


In [21]:
# 4 — Copy and inspect this one archive (no Drive search)
from zipfile import ZipFile

REQUIRED_MEMBERS = {
    'train': ('train_source1.tsv', 'train_source2.tsv',
              'train_source3.tsv', 'train_ground_truth.tsv'),
    'test': ('test_source1.tsv', 'test_source2.tsv', 'test_source3.tsv'),
}
LOCAL_ZIP = LOCAL_ROOT / 'dataset.zip'
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DATASET_ZIP.stat().st_size:
    print('Copying ZIP from Drive to local Colab storage...')
    temp_zip = LOCAL_ROOT / 'dataset.zip.partial'
    with DATASET_ZIP.open('rb') as source, temp_zip.open('wb') as destination:
        shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)
    temp_zip.replace(LOCAL_ZIP)
print('Local ZIP:', LOCAL_ZIP)

with ZipFile(LOCAL_ZIP) as archive:
    members = [info for info in archive.infolist() if not info.is_dir()]
    print('Archive contains', len(members), 'files. First paths:')
    for info in members[:20]:
        print(' ', info.filename)
    selected = {}
    for folder, names in REQUIRED_MEMBERS.items():
        for filename in names:
            matches = [info for info in members
                       if Path(info.filename).name.casefold() == filename.casefold()
                       and folder in [part.casefold() for part in Path(info.filename).parts[:-1]]]
            if len(matches) != 1:
                raise ValueError(
                    f'Expected one {folder}/{filename}, found {len(matches)}. '
                    'Inspect the printed archive paths and update REQUIRED_MEMBERS if necessary.'
                )
            selected[filename] = matches[0]
    uncompressed_bytes = sum(info.file_size for info in selected.values())
    free_bytes = shutil.disk_usage(LOCAL_ROOT).free
    missing_bytes = sum(info.file_size for name, info in selected.items()
                        if not (EXTRACT_DIR / ('train' if name.startswith('train_') else 'test') / name).is_file()
                        or (EXTRACT_DIR / ('train' if name.startswith('train_') else 'test') / name).stat().st_size != info.file_size)
    print(f'Selected TSV size: {uncompressed_bytes / 1024**3:.2f} GiB; free local disk: {free_bytes / 1024**3:.2f} GiB')
    if missing_bytes > free_bytes:
        raise OSError('Insufficient Colab local disk to extract the selected files.')
    files = {}
    for filename, info in selected.items():
        folder = 'train' if filename.startswith('train_') else 'test'
        target = EXTRACT_DIR / folder / filename
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or target.stat().st_size != info.file_size:
            print('Extracting:', info.filename)
            temp_target = target.with_name(target.name + '.partial')
            with archive.open(info) as source, temp_target.open('wb') as destination:
                shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)
            temp_target.replace(target)
        files[filename] = target
print('All seven TSV files ready.')


Local ZIP: /content/amazon_ml_challenge/dataset.zip
Archive contains 8 files. First paths:
  dataset/.DS_Store
  dataset/test/test_source1.tsv
  dataset/test/test_source2.tsv
  dataset/test/test_source3.tsv
  dataset/train/train_ground_truth.tsv
  dataset/train/train_source1.tsv
  dataset/train/train_source2.tsv
  dataset/train/train_source3.tsv
Selected TSV size: 2.35 GiB; free local disk: 80.49 GiB
All seven TSV files ready.


In [22]:
# 5 — Save unzipped TSVs to the existing Data/Processed folder in Drive.
# A fresh Colab runtime still extracts local working copies for faster analysis.
for filename, local_path in files.items():
    folder = 'train' if filename.startswith('train_') else 'test'
    saved_path = DRIVE_EXTRACT_DIR / folder / filename
    saved_path.parent.mkdir(parents=True, exist_ok=True)
    if not saved_path.is_file() or saved_path.stat().st_size != local_path.stat().st_size:
        print('Saving extracted file to Drive:', saved_path)
        pending = saved_path.with_name(saved_path.name + '.partial')
        with local_path.open('rb') as source, pending.open('wb') as destination:
            shutil.copyfileobj(source, destination, length=8 * 1024 * 1024)
        if pending.stat().st_size != local_path.stat().st_size:
            raise OSError(f'Incomplete Drive copy: {pending}')
        pending.replace(saved_path)
    print('Verified Drive copy:', saved_path)


Verified Drive copy: /content/drive/MyDrive/Amazon_Ml_Challange_2026/Data/Processed/dataset/train/train_source1.tsv
Verified Drive copy: /content/drive/MyDrive/Amazon_Ml_Challange_2026/Data/Processed/dataset/train/train_source2.tsv
Verified Drive copy: /content/drive/MyDrive/Amazon_Ml_Challange_2026/Data/Processed/dataset/train/train_source3.tsv
Verified Drive copy: /content/drive/MyDrive/Amazon_Ml_Challange_2026/Data/Processed/dataset/train/train_ground_truth.tsv
Verified Drive copy: /content/drive/MyDrive/Amazon_Ml_Challange_2026/Data/Processed/dataset/test/test_source1.tsv
Verified Drive copy: /content/drive/MyDrive/Amazon_Ml_Challange_2026/Data/Processed/dataset/test/test_source2.tsv
Verified Drive copy: /content/drive/MyDrive/Amazon_Ml_Challange_2026/Data/Processed/dataset/test/test_source3.tsv


In [23]:
# 6 — Explicit train/test paths; stop early on missing or empty files
for filename, path in files.items():
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f'Missing or empty: {path}')
    print(f'{filename}: {path} ({path.stat().st_size / 1024**2:.1f} MiB)')
TRAIN_S1, TRAIN_S2, TRAIN_S3, TRAIN_GT = (files[name] for name in REQUIRED_MEMBERS['train'])
TEST_S1, TEST_S2, TEST_S3 = (files[name] for name in REQUIRED_MEMBERS['test'])


train_source1.tsv: /content/amazon_ml_challenge/extracted/train/train_source1.tsv (200.3 MiB)
train_source2.tsv: /content/amazon_ml_challenge/extracted/train/train_source2.tsv (466.6 MiB)
train_source3.tsv: /content/amazon_ml_challenge/extracted/train/train_source3.tsv (480.4 MiB)
train_ground_truth.tsv: /content/amazon_ml_challenge/extracted/train/train_ground_truth.tsv (121.1 MiB)
test_source1.tsv: /content/amazon_ml_challenge/extracted/test/test_source1.tsv (166.9 MiB)
test_source2.tsv: /content/amazon_ml_challenge/extracted/test/test_source2.tsv (485.9 MiB)
test_source3.tsv: /content/amazon_ml_challenge/extracted/test/test_source3.tsv (482.6 MiB)


In [24]:
# 7 — File sizes BEFORE loading data

file_size_rows = []

for filename, path in files.items():
    size_bytes = path.stat().st_size
    file_size_rows.append({
        "file": filename,
        "size_mb": round(size_bytes / 1024**2, 2),
        "size_gb": round(size_bytes / 1024**3, 3),
        "path": str(path),
    })

file_sizes_df = (
    pd.DataFrame(file_size_rows)
      .sort_values("size_mb", ascending=False)
      .reset_index(drop=True)
)

display(file_sizes_df)


,file,size_mb,size_gb,path
0,test_source2.tsv,485.86,0.474,/content/amazon_ml_challenge/extracted/test/te...
1,test_source3.tsv,482.56,0.471,/content/amazon_ml_challenge/extracted/test/te...
2,train_source3.tsv,480.37,0.469,/content/amazon_ml_challenge/extracted/train/t...
3,train_source2.tsv,466.63,0.456,/content/amazon_ml_challenge/extracted/train/t...
4,train_source1.tsv,200.34,0.196,/content/amazon_ml_challenge/extracted/train/t...
5,test_source1.tsv,166.91,0.163,/content/amazon_ml_challenge/extracted/test/te...
6,train_ground_truth.tsv,121.13,0.118,/content/amazon_ml_challenge/extracted/train/t...


In [25]:
# 8 — Verify TSV parsing with only 5 rows from each file

for filename, path in files.items():
    print("\n" + "=" * 90)
    print(filename)
    print("=" * 90)

    sample = pl.read_csv(
        path,
        separator="\t",
        n_rows=5,
        infer_schema_length=1000,
    )

    print("Columns:", sample.columns)
    display(sample)



train_source1.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']


entity_id,business_name,business_address,country
str,str,str,str
"""S1-925783039""","""Orelee's Barbershop""","""1795 Westchester Drive, High P…","""US"""
"""S1-773889195""","""Prime Money""","""17560 Ellis Road, Tahlequah, O…","""US"""
"""S1-377745466""","""B+ Retail Inc""","""1712 Montebello Avenue, Phoeni…","""US"""
"""S1-133037285""","""Christ Chapel""","""2100 Cameron Drive, Unit APART…","""US"""
"""S1-755362802""","""Prabhav Business Center""","""797, Lake Town Block A, Kolkat…","""India"""



train_source2.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']


entity_id,business_name,business_address,country
str,str,str,str
"""S2-166376419""","""राम मार्केटिंग प्राइवेट लिमिटे…","""KH NO. -570/13, NEW DELHI, WES…","""India"""
"""S2-764573417""","""-- Holloway Peak Inc Seafood""","""105 ELM ST, MORGANTON, NC""","""US"""
"""S2-639257739""","""आदित्य प्रॉपर्टीज एलएलपी""","""G-3/571, GULMOHAR COLONY, BHOP…","""India"""
"""S2-163963287""","""Summit Inc""","""GREENSBORO, NC, 19 1/2 STARDUS…","""US"""
"""S2-49942811""","""Delta Tetlecommunication Inc""","""914 PIERPONT AVE, CLEVELAND, O…","""US"""



train_source3.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']


...

entity_id,business_name,business_address,country
str,str,str,str
"""S1-714132312""","""Zephay Labs Inc""","""2621 Cotten Road, Tyler, TX""","""US"""
"""S1-106407869""","""Vision Partners Corp""","""IA, Iowa City, 1064 Newton Rd,…","""US"""
"""S1-156285671""","""<< Team Ecole""","""175 Boulevard du Président Fra…","""France"""
"""S1-689823050""","""Red Perfect Trading""","""Mirzapur, Ews 12, Uttar Prades…","""India"""
"""S1-921369899""","""ZNB Club SARL""","""Nouvelle-Aquitaine, La Teste-d…","""France"""



test_source2.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']


entity_id,business_name,business_address,country
str,str,str,str
"""S2-192345572""","""Brahma Infosoft""","""COIMATORE COLONY, HUNSUR TQMYS…","""India"""
"""S2-566025912""","""Marina Ecole France Sarl""","""63 R. DE DIEPPE, LILLE, Hauts-…","""France"""
"""S2-158121477""","""SCI Ptit Àmicale""","""18 RUE JEN ZAY, Dunkerque, Nor…","""France"""
"""S2-89663826""","""Apex Summit""","""67 KENTUCKY ST, SALYERSVILLE, …","""US"""
"""S2-884102769""","""Fresh Truist""","""8264 FILLY COURT, ROANOKE COUN…","""US"""



test_source3.tsv
Columns: ['entity_id', 'business_name', 'business_address', 'country']


entity_id,business_name,business_address,country
str,str,str,str
"""S3-462677478""","""मॉडर्न फाइनेंस""","""No 10 Enkay Square, 448A, Udyo…","""India"""
"""S3-374810425""","""Shri Sai Infratech Co""","""3/115, East Delhi, DL""","""India"""
"""S3-198586129""","""Fractales Amis Groupe S.A.S""","""23 Rue Icmre, La Teste-de-buch…","""France"""
"""S3-10300249""","""Shri Supreme Consulting Privat…","""H.no 910 A 3503, Mumbai, महारा…","""India"""
"""S3-604980231""","""Prime Realty Ventures Public L…","""G.t. Karnal Road, Industrial A…","""India"""


# EDA: dataset scale, integrity, geography, singleton rate and matching structure
# The first pass scans TSVs without loading them all into Pandas. Full train frames
# are loaded later for ground-truth and blocking analysis.


In [26]:
# 9 — Memory-safe basic row counts, schema, missing values, countries

lazy_sources = {
    "Train S1": pl.scan_csv(TRAIN_S1, separator="\t", infer_schema_length=10000),
    "Train S2": pl.scan_csv(TRAIN_S2, separator="\t", infer_schema_length=10000),
    "Train S3": pl.scan_csv(TRAIN_S3, separator="\t", infer_schema_length=10000),
    "Test S1": pl.scan_csv(TEST_S1, separator="\t", infer_schema_length=10000),
    "Test S2": pl.scan_csv(TEST_S2, separator="\t", infer_schema_length=10000),
    "Test S3": pl.scan_csv(TEST_S3, separator="\t", infer_schema_length=10000),
}

lazy_gt = pl.scan_csv(TRAIN_GT, separator="\t", infer_schema_length=10000)

row_counts = {}

for name, df in {**lazy_sources, "Train GT": lazy_gt}.items():
    n = df.select(pl.len()).collect().item()
    row_counts[name] = int(n)

print("ROW COUNTS")
print("-" * 60)
for name, n in row_counts.items():
    print(f"{name:12}: {n:,}")

print("\nSource-1 columns:")
print(lazy_sources["Train S1"].collect_schema().names())

expected_source = {"entity_id", "business_name", "business_address", "country"}
for name, frame in lazy_sources.items():
    actual = set(frame.collect_schema().names())
    if actual != expected_source:
        raise ValueError(f"{name}: expected {expected_source}, found {actual}")
for name, path, prefix in [
    ("Train S1", TRAIN_S1, "S1-"), ("Train S2", TRAIN_S2, "S2-"),
    ("Train S3", TRAIN_S3, "S3-"), ("Test S1", TEST_S1, "S1-"),
    ("Test S2", TEST_S2, "S2-"), ("Test S3", TEST_S3, "S3-"),
]:
    invalid = pl.scan_csv(path, separator="\t", infer_schema_length=10000).select(
        (~pl.col("entity_id").str.starts_with(prefix)).sum()
    ).collect().item()
    print(f"{name}: invalid ID prefix = {invalid:,}")
if set(lazy_gt.collect_schema().names()) != {"source1_entity_id", "matched_entity_ids"}:
    raise ValueError(f"Unexpected ground-truth columns: {lazy_gt.collect_schema().names()}")


ROW COUNTS
------------------------------------------------------------
Train S1    : 2,206,821
Train S2    : 5,034,616
Train S3    : 5,285,603
Test S1     : 1,732,544
Test S2     : 4,887,273
Test S3     : 5,082,316
Train GT    : 2,206,821

Source-1 columns:
['entity_id', 'business_name', 'business_address', 'country']
Train S1: invalid ID prefix = 0
Train S2: invalid ID prefix = 0
Train S3: invalid ID prefix = 0
Test S1: invalid ID prefix = 0
Test S2: invalid ID prefix = 0
Test S3: invalid ID prefix = 0


In [27]:
# 10 — Missing/empty values and duplicate entity IDs

quality_rows = []

for name, df in lazy_sources.items():
    n = row_counts[name]

    result = (
        df.select([
            pl.col("entity_id").n_unique().alias("unique_ids"),

            (
                pl.col("business_name").is_null()
                | (pl.col("business_name").str.strip_chars() == "")
            ).sum().alias("empty_name"),

            (
                pl.col("business_address").is_null()
                | (pl.col("business_address").str.strip_chars() == "")
            ).sum().alias("empty_address"),

            (
                pl.col("country").is_null()
                | (pl.col("country").str.strip_chars() == "")
            ).sum().alias("empty_country"),

            (
                (pl.col("business_name").is_null() | (pl.col("business_name").str.strip_chars() == ""))
                & (pl.col("business_address").is_null() | (pl.col("business_address").str.strip_chars() == ""))
            ).sum().alias("both_name_address_empty"),
            pl.col("business_name").str.len_chars().mean().alias("mean_name_len"),
            pl.col("business_address").str.len_chars().mean().alias("mean_address_len"),
        ])
        .collect()
        .to_dicts()[0]
    )

    quality_rows.append({
        "dataset": name,
        "rows": n,
        "unique_ids": int(result["unique_ids"]),
        "duplicate_entity_ids": n - int(result["unique_ids"]),
        "empty_name": int(result["empty_name"]),
        "empty_name_pct": 100 * int(result["empty_name"]) / max(n, 1),
        "empty_address": int(result["empty_address"]),
        "empty_address_pct": 100 * int(result["empty_address"]) / max(n, 1),
        "empty_country": int(result["empty_country"]),
        "empty_country_pct": 100 * int(result["empty_country"]) / max(n, 1),
        "both_name_address_empty": int(result["both_name_address_empty"]),
        "mean_name_len": result["mean_name_len"],
        "mean_address_len": result["mean_address_len"],
    })

quality_df = pd.DataFrame(quality_rows)
display(quality_df)


,dataset,rows,unique_ids,duplicate_entity_ids,empty_name,empty_name_pct,empty_address,empty_address_pct,empty_country,empty_country_pct,both_name_address_empty,mean_name_len,mean_address_len
0,Train S1,2206821,2206821,0,0,0.0,0,0.000000,0,0.0,0,24.034403,52.066213
1,Train S2,5034616,5034616,0,0,0.0,168967,3.356105,0,0.0,0,25.103575,47.831173
2,Train S3,5285603,5285603,0,0,0.0,175916,3.328211,0,0.0,0,25.202144,48.322719
3,Test S1,1732544,1732544,0,0,0.0,0,0.000000,0,0.0,0,23.836426,57.213180
4,Test S2,4887273,4887273,0,0,0.0,129408,2.647857,0,0.0,0,25.703646,51.784587
5,Test S3,5082316,5082316,0,0,0.0,136098,2.677874,0,0.0,0,25.655194,50.081593


In [28]:
# 11 — Country distribution

country_frames = []

for name, df in lazy_sources.items():
    temp = (
        df.group_by("country")
          .agg(pl.len().alias("count"))
          .sort("count", descending=True)
          .collect()
          .with_columns(pl.lit(name).alias("dataset"))
          .select(["dataset", "country", "count"])
    )

    country_frames.append(temp)

country_df = pl.concat(country_frames)
display(country_df)
train_countries = set(country_df.filter(pl.col("dataset").str.starts_with("Train"))["country"].drop_nulls().to_list())
test_countries = set(country_df.filter(pl.col("dataset").str.starts_with("Test"))["country"].drop_nulls().to_list())
print("Training countries:", sorted(train_countries))
print("Test countries:", sorted(test_countries))
print("Unseen test countries:", sorted(test_countries - train_countries))
print("Country handling must accept any string, including test-only countries.")


dataset,country,count
str,str,u32
"""Train S1""","""US""",1323633
"""Train S1""","""India""",883188
"""Train S2""","""US""",3016817
"""Train S2""","""India""",2017799
"""Train S3""","""US""",3170056
…,…,…
"""Test S2""","""US""",1871330
"""Test S2""","""France""",703378
"""Test S3""","""India""",2405000


Training countries: ['India', 'US']
Test countries: ['France', 'India', 'US']
Unseen test countries: ['France']
Country handling must accept any string, including test-only countries.


In [29]:
# 12 — Load full training data with Pandas for ground-truth and blocking analysis
# This is the first cell that loads the full TRAIN sources into RAM.
# If this crashes due to RAM, stop here and tell ChatGPT your file sizes/row counts.

def load_tsv(path):
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)

def parse_ids(cell):
    cell = (cell or "").strip()
    if not cell:
        return []
    return [x.strip() for x in cell.split(",") if x.strip()]

print("Loading full training files into RAM...")

s1 = load_tsv(TRAIN_S1)
s2 = load_tsv(TRAIN_S2)
s3 = load_tsv(TRAIN_S3)
gt = load_tsv(TRAIN_GT)

print(f"train s1={len(s1):,}  s2={len(s2):,}  s3={len(s3):,}  gt={len(gt):,}")

gt["matches"] = gt["matched_entity_ids"].map(parse_ids)
gt["n_matches"] = gt["matches"].map(len)

# sanity: every GT row should reference a real S1
s1_ids = set(s1["entity_id"])
unknown_gt_s1 = sum(
    1 for x in gt["source1_entity_id"]
    if x not in s1_ids
)

print("GT rows with unknown source1_entity_id:", unknown_gt_s1)


Loading full training files into RAM...
train s1=2,206,821  s2=5,034,616  s3=5,285,603  gt=2,206,821
GT rows with unknown source1_entity_id: 0


In [30]:
# 13 — Check ground-truth coverage and referenced target IDs.
valid_target_ids = set(s2["entity_id"]) | set(s3["entity_id"])
missing_s1 = set(s1["entity_id"]) - set(gt["source1_entity_id"])
repeated_s1 = int(gt["source1_entity_id"].duplicated().sum())
unknown_target_ids = {mid for matches in gt["matches"] for mid in matches
                      if mid not in valid_target_ids}
invalid_target_prefixes = {mid for matches in gt["matches"] for mid in matches
                           if not mid.startswith(("S2-", "S3-"))}
print("S1 IDs without ground-truth row:", len(missing_s1))
print("Repeated ground-truth S1 IDs:", repeated_s1)
print("Unknown target IDs:", len(unknown_target_ids))
print("Wrong target prefixes:", len(invalid_target_prefixes))
if missing_s1 or repeated_s1 or unknown_target_ids or invalid_target_prefixes:
    print("Review these integrity warnings before using ground truth as labels.")


S1 IDs without ground-truth row: 0
Repeated ground-truth S1 IDs: 0
Unknown target IDs: 0
Wrong target prefixes: 0


In [31]:
# 14 — Q1: Search space

print("=" * 75)
print("Q1: SEARCH SPACE — HOW HARD MUST BLOCKING WORK?")
print("=" * 75)

pairs_per_s1_train = len(s2) + len(s3)

brute_train = len(s1) * pairs_per_s1_train
brute_test = (
    row_counts["Test S1"]
    * (row_counts["Test S2"] + row_counts["Test S3"])
)

print(f"Brute-force candidates per train S1 : {pairs_per_s1_train:,}")
print(f"Brute-force pairs total (train)     : {brute_train:,}")
print(f"Brute-force pairs total (test)      : {brute_test:,}")

for k in [10, 20, 40, 80, 100]:
    rr = 1 - k / max(pairs_per_s1_train, 1)
    print(f"If TOP_K={k:3d}: train reduction ratio ≈ {rr:.8f}")


Q1: SEARCH SPACE — HOW HARD MUST BLOCKING WORK?
Brute-force candidates per train S1 : 10,320,219
Brute-force pairs total (train)     : 22,774,876,013,799
Brute-force pairs total (test)      : 17,272,751,604,416
If TOP_K= 10: train reduction ratio ≈ 0.99999903
If TOP_K= 20: train reduction ratio ≈ 0.99999806
If TOP_K= 40: train reduction ratio ≈ 0.99999612
If TOP_K= 80: train reduction ratio ≈ 0.99999225
If TOP_K=100: train reduction ratio ≈ 0.99999031


In [32]:
# 15 — Q2: Singleton and match-count distribution

print("=" * 75)
print("Q2: SINGLETON / MATCH-COUNT DISTRIBUTION")
print("=" * 75)

n_singleton = int((gt["n_matches"] == 0).sum())
n_matched = len(gt) - n_singleton

dist = Counter(gt["n_matches"])

print(f"Entities with 0 matches : {n_singleton:,} ({n_singleton/len(gt):.2%})")
print(f"Entities with >=1 match : {n_matched:,} ({n_matched/len(gt):.2%})")
print()

print("Matches per entity:")
for k in sorted(dist)[:15]:
    print(f"  {k:>3} matches : {dist[k]:>9,} ({dist[k]/len(gt):.2%})")

if dist and max(dist) > 14:
    tail = sum(v for k, v in dist.items() if k > 14)
    print(f"  >14 matches : {tail:>9,} ({tail/len(gt):.2%})")

print()
print(
    f"Mean={gt['n_matches'].mean():.3f}  "
    f"Median={gt['n_matches'].median():.0f}  "
    f"Max={gt['n_matches'].max()}"
)


Q2: SINGLETON / MATCH-COUNT DISTRIBUTION
Entities with 0 matches : 123,247 (5.58%)
Entities with >=1 match : 2,083,574 (94.42%)

Matches per entity:
    0 matches :   123,247 (5.58%)
    1 matches :   119,157 (5.40%)
    2 matches :   375,212 (17.00%)
    3 matches :   530,841 (24.05%)
    4 matches :   484,115 (21.94%)
    5 matches :   321,957 (14.59%)
    6 matches :   164,868 (7.47%)
    7 matches :    63,968 (2.90%)
    8 matches :    18,680 (0.85%)
    9 matches :     4,205 (0.19%)
   10 matches :       534 (0.02%)
   11 matches :        37 (0.00%)

Mean=3.461  Median=3  Max=11


In [33]:
# 16 — Q3: Are true matches always within the same country?

print("=" * 75)
print("Q3: CROSS-COUNTRY TRUE MATCHES")
print("=" * 75)

country_lookup = {}

for df in (s1, s2, s3):
    country_lookup.update(dict(zip(df["entity_id"], df["country"])))

same_c = 0
diff_c = 0
unknown = 0
cross_country_examples = []

for s1id, matches in zip(gt["source1_entity_id"], gt["matches"]):
    c1 = country_lookup.get(s1id)

    for m in matches:
        c2 = country_lookup.get(m)

        if c1 is None or c2 is None:
            unknown += 1
        elif c1 == c2:
            same_c += 1
        else:
            diff_c += 1

            if len(cross_country_examples) < 10:
                cross_country_examples.append((s1id, c1, m, c2))

total_true_pairs = same_c + diff_c + unknown

print(f"Total true pairs      : {total_true_pairs:,}")
print(f"Same country          : {same_c:,}")
print(f"Different country     : {diff_c:,}")
print(f"ID not found          : {unknown:,}")

if cross_country_examples:
    print("\nExamples:")
    for e in cross_country_examples:
        print(" ", e)

print()

if diff_c == 0:
    print("TRAIN EVIDENCE: no cross-country true pair was observed.")
    print("A same-country blocking route therefore preserves all observed train positives.")
else:
    print(
        f"WARNING: {diff_c/max(total_true_pairs,1):.3%} of observed true pairs "
        "cross countries."
    )
    print("Strict country blocking would lose those positives.")


Q3: CROSS-COUNTRY TRUE MATCHES
Total true pairs      : 7,638,365
Same country          : 7,638,365
Different country     : 0
ID not found          : 0

TRAIN EVIDENCE: no cross-country true pair was observed.
A same-country blocking route therefore preserves all observed train positives.


In [34]:
# 17 — Q4: Multiple matches from the same source

print("=" * 75)
print("Q4: MULTIPLE MATCHES FROM THE SAME SOURCE")
print("=" * 75)

multi_s2 = 0
multi_s3 = 0
both_sources = 0

for matches in gt["matches"]:
    n2 = sum(m.startswith("S2-") for m in matches)
    n3 = sum(m.startswith("S3-") for m in matches)

    multi_s2 += int(n2 > 1)
    multi_s3 += int(n3 > 1)
    both_sources += int(n2 >= 1 and n3 >= 1)

nz = max(int((gt["n_matches"] > 0).sum()), 1)

print(f"Entities matching >1 S2 record : {multi_s2:,} ({multi_s2/nz:.2%} of matched)")
print(f"Entities matching >1 S3 record : {multi_s3:,} ({multi_s3/nz:.2%} of matched)")
print(f"Entities matching both sources : {both_sources:,} ({both_sources/nz:.2%} of matched)")

all_matched = [m for matches in gt["matches"] for m in matches]

print("Matched IDs by source:", Counter(x[:2] for x in all_matched))

reused = sum(
    1 for _, count in Counter(all_matched).items()
    if count > 1
)

print("S2/S3 records matched by >1 S1 entity:", reused)


Q4: MULTIPLE MATCHES FROM THE SAME SOURCE
Entities matching >1 S2 record : 1,129,968 (54.23% of matched)
Entities matching >1 S3 record : 1,224,128 (58.75% of matched)
Entities matching both sources : 1,776,047 (85.24% of matched)
Matched IDs by source: Counter({'S3': 3944746, 'S2': 3693619})
S2/S3 records matched by >1 S1 entity: 0


In [35]:
# 18 — Create positive-pair table

positive_pairs = (
    gt.loc[gt["n_matches"] > 0, ["source1_entity_id", "matches"]]
      .explode("matches")
      .rename(columns={"matches": "matched_entity_id"})
      .reset_index(drop=True)
)

positive_pairs["label"] = 1

print("Positive pairs:", len(positive_pairs))
display(positive_pairs.head(20))


Positive pairs: 7638365


,source1_entity_id,matched_entity_id,label
0,S1-965667,S2-681193310,1
1,S1-965667,S2-743505751,1
2,S1-965667,S3-775321672,1
3,S1-965667,S3-11291185,1
4,S1-965667,S3-860443364,1
5,S1-55344266,S2-249013014,1
6,S1-55344266,S2-197070651,1
7,S1-55344266,S3-478195123,1
8,S1-55344266,S3-384364074,1
9,S1-343815751,S2-790675320,1


In [36]:
# 19 — Sample true pairs and singletons
# Read these manually. They tell you which normalizations/features are useful.

record_lookup = {}

for df in (s1, s2, s3):
    for r in df.itertuples(index=False):
        record_lookup[r.entity_id] = (
            r.business_name,
            r.business_address,
            r.country,
        )

print("=" * 75)
print("SAMPLE TRUE PAIRS")
print("=" * 75)

shown = 0

for s1id, matches in zip(gt["source1_entity_id"], gt["matches"]):
    if not matches or s1id not in record_lookup:
        continue

    name1, addr1, country1 = record_lookup[s1id]

    print(f"\n[{s1id}] ({country1})")
    print("  NAME:", name1)
    print("  ADDR:", addr1)

    for m in matches:
        if m in record_lookup:
            name2, addr2, country2 = record_lookup[m]
            print(f"    -> [{m}] ({country2})")
            print("       NAME:", name2)
            print("       ADDR:", addr2)

    shown += 1
    if shown >= 12:
        break

print("\n" + "=" * 75)
print("SAMPLE SINGLETONS")
print("=" * 75)

shown = 0

for s1id, matches in zip(gt["source1_entity_id"], gt["matches"]):
    if matches or s1id not in record_lookup:
        continue

    name1, addr1, country1 = record_lookup[s1id]
    print(f"[{s1id}] ({country1}) {name1} | {addr1}")

    shown += 1
    if shown >= 8:
        break


SAMPLE TRUE PAIRS

[S1-965667] (US)
  NAME: Maure Williams Colombier Inc
  ADDR: 85 Wayne Avenue, Ticonderoga, NY
    -> [S2-681193310] (US)
       NAME: Maure Wilblims Colombier Inc
       ADDR: 
    -> [S2-743505751] (US)
       NAME: Maure Williams Colombier
       ADDR: 
    -> [S3-775321672] (US)
       NAME: Dréxkor
       ADDR: 85 Wanye Avenue, Ticonderoga Townshiip, New York
    -> [S3-11291185] (US)
       NAME: maurewilliamscolombier.com
       ADDR: Wayne Ave, Ticonderoga Townshiip, New York
    -> [S3-860443364] (US)
       NAME: Maure Williams Inc Center
       ADDR: 

[S1-55344266] (India)
  NAME: Raj Investments LLP
  ADDR: 6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu
    -> [S2-249013014] (India)
       NAME: ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி
       ADDR: 6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu
    -> [S2-197070651] (India)
       NAME: Raj Investments LLP
       ADDR: 6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Ta

## True-pair difficulty and blocking clues

These checks focus on positive pairs and a bounded sample of name/address tokens. They help decide which exact rules work and which true pairs need fuzzy or alternative blocking routes.


In [37]:
# 20 — Analyze all known positive pairs without building a giant cross join.
from rapidfuzz import fuzz

def simple_norm(value):
    value = unicodedata.normalize("NFKC", str(value)).casefold().replace("&", " and ")
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", value)).strip()

similarity_rows = []
for source1_id, target_id in positive_pairs[["source1_entity_id", "matched_entity_id"]].itertuples(index=False, name=None):
    left, right = record_lookup.get(source1_id), record_lookup.get(target_id)
    if left is None or right is None:
        continue
    left_name, left_addr = simple_norm(left[0]), simple_norm(left[1])
    right_name, right_addr = simple_norm(right[0]), simple_norm(right[1])
    similarity_rows.append({
        "source1_entity_id": source1_id, "matched_entity_id": target_id,
        "exact_name": bool(left_name) and left_name == right_name,
        "exact_address": bool(left_addr) and left_addr == right_addr,
        "name_token_set": fuzz.token_set_ratio(left_name, right_name),
        "name_token_sort": fuzz.token_sort_ratio(left_name, right_name),
        "address_ratio": fuzz.ratio(left_addr, right_addr),
    })
true_similarity = pd.DataFrame(similarity_rows)
print("Resolvable true pairs:", len(true_similarity), "of", len(positive_pairs))
if not true_similarity.empty:
    display(true_similarity[["exact_name", "exact_address"]].mean().to_frame("share"))
    display(true_similarity[["name_token_set", "name_token_sort", "address_ratio"]]
            .describe(percentiles=[.05, .25, .5, .75, .95]))
    print("Hardest true pairs by name similarity:")
    hard_pairs = true_similarity.nsmallest(15, ["name_token_set", "address_ratio"])
    for row in hard_pairs.itertuples(index=False):
        left, right = record_lookup[row.source1_entity_id], record_lookup[row.matched_entity_id]
        print(f"{row.source1_entity_id} -> {row.matched_entity_id} | name={row.name_token_set:.1f}, addr={row.address_ratio:.1f}")
        print("  S1:", left[0], "|", left[1])
        print("  MATCH:", right[0], "|", right[1])


Resolvable true pairs: 7638365 of 7638365


,share
exact_name,0.217109
exact_address,0.082658


,name_token_set,name_token_sort,address_ratio
count,7.638365e+06,7.638365e+06,7.638365e+06
mean,8.657113e+01,8.019767e+01,7.559015e+01
std,2.514829e+01,2.511832e+01,2.332902e+01
min,0.000000e+00,0.000000e+00,0.000000e+00
5%,1.250000e+01,1.111111e+01,3.000000e+01
25%,8.750000e+01,7.368421e+01,6.551724e+01
50%,1.000000e+02,8.947368e+01,8.288288e+01
75%,1.000000e+02,1.000000e+02,9.180328e+01
95%,1.000000e+02,1.000000e+02,1.000000e+02
max,1.000000e+02,1.000000e+02,1.000000e+02


Hardest true pairs by name similarity:
S1-77502115 -> S3-88949956 | name=0.0, addr=24.2
  S1: Tech Tech | Madhya Pradesh, Plot No.6/1, Old Palasiya (Gauri Tower), Opp. Barwani Plaza, Indore
  MATCH: टेक टेक | C-6/1, Indore, MP
S1-77502115 -> S3-717039560 | name=0.0, addr=27.7
  S1: Tech Tech | Madhya Pradesh, Plot No.6/1, Old Palasiya (Gauri Tower), Opp. Barwani Plaza, Indore
  MATCH: टेक टेक | No C-6/1, Indore, MP
S1-406865257 -> S3-901091678 | name=0.0, addr=31.7
  S1: International International | 6Th Floor, Manglam Fun Square Mall, Durga Nursery Road, Shakti Nagar, Rajasthan, Office No. 616, Udaipur, Girwa
  MATCH: इंटरनेशनल इंटरनेशनल | Office No. 616, Girwa, Udaipur, राजस्थान
S1-77502115 -> S3-753440478 | name=0.0, addr=32.3
  S1: Tech Tech | Madhya Pradesh, Plot No.6/1, Old Palasiya (Gauri Tower), Opp. Barwani Plaza, Indore
  MATCH: टेक टेक | Door No C-6/1, Indore, MP
S1-573104988 -> S3-102991195 | name=0.0, addr=34.6
  S1: Lanco & Sons Pvt Ltd | Shukan Status, Opp.Shukan Residen

In [38]:
# 21 — Frequent name/address tokens from a bounded reproducible sample.
# This avoids multiplying RAM use by duplicating all source strings.
TOKEN_SAMPLE_PER_SOURCE = 50_000
name_counts, address_counts = Counter(), Counter()
for source in (s1, s2, s3):
    subset = source.sample(n=min(TOKEN_SAMPLE_PER_SOURCE, len(source)), random_state=42)
    for value in subset["business_name"]:
        name_counts.update(set(simple_norm(value).split()))
    for value in subset["business_address"]:
        address_counts.update(set(simple_norm(value).split()))
print("Common name tokens:", name_counts.most_common(25))
print("Common address tokens:", address_counts.most_common(25))


Common name tokens: [('limited', 23368), ('private', 21000), ('llc', 18627), ('inc', 13272), ('ltd', 11482), ('and', 10335), ('pvt', 6732), ('center', 4853), ('com', 4020), ('s', 4014), ('partners', 3873), ('services', 3719), ('group', 3610), ('ल', 3575), ('corp', 3553), ('ट', 3518), ('india', 3490), ('र', 3426), ('c', 3398), ('co', 3264), ('ड', 3152), ('प', 3114), ('म', 3102), ('care', 3035), ('of', 2630)]
Common address tokens: [('no', 25663), ('road', 22705), ('street', 13832), ('city', 9700), ('drive', 9510), ('floor', 9509), ('new', 8898), ('nagar', 8852), ('1', 8808), ('avenue', 8289), ('delhi', 7845), ('maharashtra', 7719), ('c', 7320), ('a', 7302), ('st', 6891), ('rd', 6884), ('west', 6838), ('2', 6669), ('unit', 6470), ('tx', 6029), ('dr', 5766), ('b', 5513), ('mumbai', 5458), ('plot', 5298), ('pradesh', 5023)]


In [39]:
# 22 — Save EDA outputs to Drive

REPORT_DIR = DRIVE_PROJECT / "reports"
PROCESSED_DIR = DRIVE_PROJECT / "processed"

file_sizes_df.to_csv(REPORT_DIR / "file_sizes.csv", index=False)
quality_df.to_csv(REPORT_DIR / "source_quality_all.csv", index=False)
country_df.write_csv(REPORT_DIR / "country_distribution.csv")

match_distribution_df = pd.DataFrame(
    sorted(dist.items()),
    columns=["num_matches", "entities"]
)
match_distribution_df["percent"] = (
    100 * match_distribution_df["entities"] / len(gt)
)
match_distribution_df.to_csv(
    REPORT_DIR / "match_distribution.csv",
    index=False
)

cross_country_df = pd.DataFrame(
    cross_country_examples,
    columns=["source1_entity_id", "s1_country", "matched_entity_id", "match_country"]
)
cross_country_df.to_csv(
    REPORT_DIR / "cross_country_examples.csv",
    index=False
)

positive_pairs.to_parquet(
    PROCESSED_DIR / "positive_pairs.parquet",
    index=False
)

print("EDA reports saved to:", REPORT_DIR)
print("Positive pairs saved to:", PROCESSED_DIR / "positive_pairs.parquet")


EDA reports saved to: /content/drive/MyDrive/Amazon_ML_Challenge/reports
Positive pairs saved to: /content/drive/MyDrive/Amazon_ML_Challenge/processed/positive_pairs.parquet
